# 🔢 DígitoVision — Classificação de Dígitos Manuscritos (MNIST)

**Autor:** Rian Gomes · **Curso:** Desenvolvimento de IA para Análise Preditiva [T1] — SCTEC
**Módulo 2 — Mini-Projeto Avaliativo**

---

## 🎯 Objetivo do sistema

Construir um **pipeline preditivo multiclasse ponta a ponta** que reconhece **dígitos manuscritos (0 a 9)**
a partir do dataset **MNIST**, e então:

1. **Compara 3 modelos** de Machine Learning (2 clássicos + 1 rede neural);
2. **Estressa** os modelos com dados que eles **nunca viram** (teste *Out-of-Distribution*);
3. **Prediz dígitos escritos à mão** pelo próprio autor (imagem real fotografada/desenhada).

## 🗺️ Roteiro do notebook

| Fase | Conteúdo |
|---|---|
| **1** | Carregamento e Análise Exploratória (EDA) |
| **2** | Pré-processamento e divisão estratificada dos dados |
| **3** | Treinamento dos 3 modelos (Random Forest, KNN, MLP) |
| **4** | Avaliação comparativa (matrizes de confusão + métricas) |
| **5.1 / 5.2** | Robustez: classes ocultas + inferência *Out-of-Distribution* |
| **5.3** | Inferência com imagem manuscrita própria |

> **Como executar:** criar um ambiente Python 3.11, instalar `pip install -r requirements.txt` e
> executar as células na ordem. O download do MNIST (~15 MB) acontece automaticamente na Fase 1.

## ⚙️ Setup — importação das bibliotecas

Importamos tudo que será usado no projeto e fixamos as **sementes aleatórias** para que os
resultados sejam **reprodutíveis** (rodar de novo dá o mesmo resultado).

In [ ]:
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)
import joblib

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reprodutibilidade
SEMENTE = 42
np.random.seed(SEMENTE)
tf.random.set_seed(SEMENTE)

sns.set_theme(style="whitegrid")

# Caminhos RELATIVOS (o enunciado proíbe caminhos absolutos).
# Funciona tanto rodando a partir da raiz do repositório quanto da pasta notebook/.
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebook":
    BASE_DIR = BASE_DIR.parent
DIR_DADOS   = BASE_DIR / "data" / "meus_digitos"
DIR_GRAF    = BASE_DIR / "outputs" / "graficos"
DIR_MODELOS = BASE_DIR / "outputs" / "modelos"
for d in (DIR_DADOS, DIR_GRAF, DIR_MODELOS):
    d.mkdir(parents=True, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("Ambiente pronto. Base do projeto:", BASE_DIR)

---
# 🔍 Fase 1 — Carregamento e Análise Exploratória de Imagens (EDA)

Nesta fase carregamos o MNIST, olhamos a **estrutura dos dados** (quantas imagens, qual formato),
conferimos se as classes estão **balanceadas** e visualizamos exemplos de cada dígito.

In [ ]:
# Baixando o MNIST oficial do OpenML (na 1ª vez demora ~40s; depois fica em cache)
print("Baixando o dataset MNIST...")
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")

# Separando entradas (X = pixels) e rótulos (y = o dígito de cada imagem)
X, y = mnist.data, mnist.target
# O target vem como texto ('0','1',...); convertemos para inteiro
y = y.astype(np.uint8)
print("Finalizou o download!")

In [ ]:
# Dimensionalidade das matrizes
print(f"Formato de X (amostras, pixels): {X.shape}")   # (70000, 784)
print(f"Formato de y (rótulos):          {y.shape}")   # (70000,)
print(f"Classes existentes:              {np.unique(y)}")
print(f"Valor mínimo e máximo de pixel:  {X.min():.0f} a {X.max():.0f}")

### Distribuição das classes

Precisamos verificar se há aproximadamente a **mesma quantidade** de cada dígito. Um dataset
muito desbalanceado enviesaria os modelos (e as métricas).

In [ ]:
contagem = pd.Series(y).value_counts().sort_index()
print("Quantidade de imagens por dígito:")
print(contagem)

plt.figure(figsize=(9, 4))
ax = sns.barplot(x=contagem.index, y=contagem.values, color="#4C72B0")
ax.set_title("Distribuição das classes no MNIST (0 a 9)")
ax.set_xlabel("Dígito")
ax.set_ylabel("Quantidade de imagens")
for i, v in enumerate(contagem.values):
    ax.text(i, v + 200, str(v), ha="center", fontsize=8)
plt.tight_layout()
plt.savefig(DIR_GRAF / "fase1_distribuicao_classes.png", dpi=120)
plt.show()

print(f"\nMédia de imagens por classe: {contagem.mean():.0f}")
print(f"Classe menos frequente: dígito {contagem.idxmin()} ({contagem.min()} imgs)")
print(f"Classe mais frequente:  dígito {contagem.idxmax()} ({contagem.max()} imgs)")

### Grade visual — um exemplo de cada dígito

Para desenhar uma imagem, pegamos o vetor de 784 números e o transformamos de volta em uma
matriz 28×28 com `.reshape(28, 28)`.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for digito, ax in enumerate(axes.flat):
    # pega a primeira imagem cujo rótulo é 'digito'
    idx = np.where(y == digito)[0][0]
    ax.imshow(X[idx].reshape(28, 28), cmap="binary")
    ax.set_title(f"Dígito: {digito}", fontsize=11)
    ax.axis("off")
plt.suptitle("Um exemplo de cada dígito (0–9) — grade 2×5", fontsize=14)
plt.tight_layout()
plt.savefig(DIR_GRAF / "fase1_grade_digitos.png", dpi=120)
plt.show()

### 🧠 Interpretação da estrutura dos dados

- **Escala de intensidade (0 a 255):** cada pixel é um número inteiro que mede a intensidade da
  tinta. `0` = fundo preto (sem tinta) e `255` = traço mais forte da caneta. Os valores
  intermediários são tons de cinza.
- **De imagem 2D para vetor de 784 *features*:** cada imagem tem $28 \times 28 = 784$ pixels.
  Os modelos clássicos (Random Forest, KNN) **não entendem uma matriz 2D** — por isso a imagem é
  "achatada" (*flatten*) em **uma linha com 784 colunas**, onde cada coluna é um pixel.
- **Consequência importante:** ao achatar, o modelo clássico trata o pixel 1 e o pixel 784 como
  variáveis isoladas e **perde a noção de vizinhança espacial**. É a diferença entre "ver o pixel"
  (clássico) e "ver o contexto" (redes mais avançadas). Vamos observar esse efeito nas métricas.

---
# 🧪 Fase 2 — Pré-processamento e Divisão dos Dados

Duas etapas essenciais antes de treinar:
1. **Divisão estratificada** em Treino / Validação / Teste (mantendo a proporção das classes);
2. **Normalização** dos pixels para o intervalo [0, 1].

### Divisão estratificada (70% treino · 10% validação · 20% teste)

Usamos `stratify=y` para garantir que **cada dígito apareça na mesma proporção** nos três conjuntos.
Fazemos em dois passos:
1. Separa **20% para teste** (nunca usado no treino → evita *data leakage*);
2. Do restante, separa ~12,5% para **validação** (≈10% do total).

In [ ]:
# Passo 1: 80% (treino+validação) e 20% teste
X_tmp, X_teste, y_tmp, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=SEMENTE, stratify=y
)
# Passo 2: do que sobrou, tira ~12,5% para validação (=10% do total)
X_treino, X_val, y_treino, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.125, random_state=SEMENTE, stratify=y_tmp
)

print(f"Treino:    {X_treino.shape[0]:>6} imagens")
print(f"Validação: {X_val.shape[0]:>6} imagens")
print(f"Teste:     {X_teste.shape[0]:>6} imagens")

# Conferindo a estratificação (proporção de cada dígito deve ser parecida nos 3 conjuntos)
prop = pd.DataFrame({
    "treino":    pd.Series(y_treino).value_counts(normalize=True).sort_index(),
    "validação": pd.Series(y_val).value_counts(normalize=True).sort_index(),
    "teste":     pd.Series(y_teste).value_counts(normalize=True).sort_index(),
}).round(3)
print("\nProporção de cada dígito por conjunto (deve ser semelhante):")
print(prop)

In [ ]:
# Normalização: pixels de [0, 255] para [0.0, 1.0], dividindo por 255
X_treino = X_treino / 255.0
X_val    = X_val / 255.0
X_teste  = X_teste / 255.0

print("Após a normalização:")
print(f"  mínimo = {X_treino.min():.1f} | máximo = {X_treino.max():.1f}")

### 🧠 Por que normalizar?

- **Modelos baseados em distância (KNN):** o KNN mede a distância entre imagens. Sem normalizar,
  os pixels de valor alto (0–255) dominariam o cálculo. Colocando tudo em [0, 1], todas as
  *features* passam a contribuir na mesma escala.
- **Redes neurais (MLP):** valores grandes deixam o treino instável e lento. Entradas em [0, 1]
  ajudam o **gradiente a convergir** mais rápido e de forma estável.
- **Random Forest:** por ser baseado em *thresholds* (cortes), é indiferente à escala — mas
  normalizamos todo mundo por consistência do pipeline.

---
# 🤖 Fase 3 — Implementação e Treinamento dos 3 Modelos

Treinamos **3 modelos distintos**, cada um com **pelo menos 2 hiperparâmetros ajustados e
justificados**. Guardamos o **tempo de treino** de cada um (usado na Fase 4).

| Modelo | Tipo | Hiperparâmetros ajustados |
|---|---|---|
| **Random Forest** | Clássico (ensemble de árvores) | `n_estimators=200`, `max_depth=20` |
| **KNN** | Clássico (baseado em distância) | `n_neighbors=3`, `weights='distance'` |
| **MLP** | Rede Neural (Keras) | `learning_rate=0.001`, arquitetura `128→64` + `epochs=15` |

Vamos guardar tudo num dicionário `resultados` para comparar depois.

In [ ]:
resultados = {}   # nome -> dict com modelo, tempo, y_pred...

### 3.1 — Random Forest

**Justificativa dos hiperparâmetros:**
- `n_estimators=200`: número de árvores. Mais árvores = previsão mais estável (menos variância).
  200 equilibra desempenho e custo.
- `max_depth=20`: profundidade máxima de cada árvore. Limita o **overfitting** (árvore decorar o
  treino) e acelera o treino, sem perder capacidade para 784 features.

In [ ]:
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    random_state=SEMENTE,
    n_jobs=-1,            # usa todos os núcleos do processador
)
rf.fit(X_treino, y_treino)
tempo_rf = time.time() - t0

y_pred_rf = rf.predict(X_teste)
acc_rf = accuracy_score(y_teste, y_pred_rf)
print(f"Random Forest treinado em {tempo_rf:.1f}s | acurácia no teste: {acc_rf:.4f}")

resultados["Random Forest"] = {"modelo": rf, "tempo": tempo_rf, "y_pred": y_pred_rf}
joblib.dump(rf, DIR_MODELOS / "random_forest.joblib")

### 3.2 — KNN (K-Nearest Neighbors)

**Justificativa dos hiperparâmetros:**
- `n_neighbors=3`: cada imagem é classificada pelo voto dos 3 vizinhos mais próximos. Valor baixo
  capta bem os padrões locais dos dígitos; muito alto "borraria" a fronteira entre classes.
- `weights='distance'`: vizinhos **mais próximos pesam mais** no voto — coerente com o fato de
  termos normalizado os pixels (Fase 2).

In [ ]:
t0 = time.time()
knn = KNeighborsClassifier(n_neighbors=3, weights="distance", n_jobs=-1)
knn.fit(X_treino, y_treino)          # KNN não "treina", só memoriza os dados
y_pred_knn = knn.predict(X_teste)    # o custo está aqui, na predição
tempo_knn = time.time() - t0

acc_knn = accuracy_score(y_teste, y_pred_knn)
print(f"KNN (fit+predict) em {tempo_knn:.1f}s | acurácia no teste: {acc_knn:.4f}")

resultados["KNN"] = {"modelo": knn, "tempo": tempo_knn, "y_pred": y_pred_knn}
joblib.dump(knn, DIR_MODELOS / "knn.joblib")

### 3.3 — MLP (Rede Neural — Perceptron Multicamadas, via Keras)

Arquitetura: `784 (entrada) → 128 → 64 → 10 (saída softmax)`.

**Justificativa dos hiperparâmetros:**
- **Nº de neurônios / camadas (128 e 64):** duas camadas ocultas que aprendem padrões cada vez mais
  abstratos dos 784 pixels. Afunilar (128→64) força a rede a resumir a informação.
- `learning_rate=0.001` (otimizador Adam): passo do gradiente. 0,001 é um valor estável e clássico
  — grande demais diverge, pequeno demais treina devagar.
- `epochs=15`, `batch_size=64`: 15 passadas pelos dados, processando 64 imagens por vez.

A camada de saída tem **10 neurônios com `softmax`** (uma probabilidade por dígito, somando 100%).

In [ ]:
mlp = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
], name="MLP_MNIST")

mlp.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",   # bom quando y é inteiro (0..9)
    metrics=["accuracy"],
)
mlp.summary()

In [ ]:
t0 = time.time()
historico = mlp.fit(
    X_treino, y_treino,
    validation_data=(X_val, y_val),   # acompanhamos o desempenho na validação
    epochs=15,
    batch_size=64,
    verbose=2,
)
tempo_mlp = time.time() - t0

# A rede devolve 10 probabilidades por imagem; argmax pega a classe mais provável
proba_mlp = mlp.predict(X_teste, verbose=0)
y_pred_mlp = np.argmax(proba_mlp, axis=1)
acc_mlp = accuracy_score(y_teste, y_pred_mlp)
print(f"\nMLP treinada em {tempo_mlp:.1f}s | acurácia no teste: {acc_mlp:.4f}")

resultados["MLP (Rede Neural)"] = {"modelo": mlp, "tempo": tempo_mlp, "y_pred": y_pred_mlp}
mlp.save(DIR_MODELOS / "mlp_mnist.keras")

### Curvas de treino da MLP

Acompanhar a perda (*loss*) e a acurácia por época ajuda a ver se a rede está aprendendo e se há
**overfitting** (quando a validação piora enquanto o treino melhora).

In [ ]:
hist = historico.history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist["loss"], label="treino")
ax1.plot(hist["val_loss"], label="validação")
ax1.set_title("Perda (loss) por época"); ax1.set_xlabel("época"); ax1.legend()
ax2.plot(hist["accuracy"], label="treino")
ax2.plot(hist["val_accuracy"], label="validação")
ax2.set_title("Acurácia por época"); ax2.set_xlabel("época"); ax2.legend()
plt.tight_layout()
plt.savefig(DIR_GRAF / "fase3_curvas_treino_mlp.png", dpi=120)
plt.show()